In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
pd.set_option('display.max_columns', None)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer



In [6]:
df = pd.read_csv('C:\\Users\\rashid\\Desktop\\customer_churn_prediction\\data\\processed\\cleaned_data.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [10]:
y = df['Churn'].map({'Yes': 1, 'No': 0})
x = df.drop(columns=['Churn', 'customerID'])

print(f"Shape of x: {x.shape}")
print(f"Shape of y: {y.shape}")
print(f'First 5 rows of y:\n{y.head()}')

Shape of x: (7043, 19)
Shape of y: (7043,)
First 5 rows of y:
0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64


In [11]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # Hər iki hissədə Churn faizinin eyni qalmasını təmin edir
)

# 2. Sütun qruplarını müəyyən edirik
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
# Rəqəmsal olmayan digər bütün sütunlar kateqoriyalıdır
cat_cols = [col for col in x.columns if col not in num_cols]

# 3. ColumnTransformer yaradırıq (Eyni anda scaling və encoding)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols) 
        # drop='first' -> multikollinerarlığın qarşısını almaq üçün dummy variable-dan birini atır
    ]
)

# 4. Preprocessing transformatorunu quraşdırırıq və tətbiq edirik
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

# 5. Yeni sütun adlarını əldə edirik (Nəticəni DataFrame kimi görmək üçün)
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_cols = cat_encoder.get_feature_names_out(cat_cols)
all_features = num_cols + list(encoded_cat_cols)

# 6. Emal olunmuş dataları DataFrame formatına gətiririk
x_train_final = pd.DataFrame(x_train_processed, columns=all_features)
x_test_final = pd.DataFrame(x_test_processed, columns=all_features)

print(f"Orijinal x sütun sayı: {x.shape[1]}")
print(f"Preprocessing-dən sonra sütun sayı: {x_train_final.shape[1]}")
print(f"\nTrain datanın ilk 3 sətiri (Hazır model üçün):")
x_train_final.head(3)

Orijinal x sütun sayı: 19
Preprocessing-dən sonra sütun sayı: 30

Train datanın ilk 3 sətiri (Hazır model üçün):


,tenure,MonthlyCharges,TotalCharges,gender_Male,SeniorCitizen_1,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0.102371,-0.521976,-0.262257,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.711743,0.337478,-0.503635,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.793155,-0.809013,-0.749883,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [12]:
x_train_final.to_csv('C:\\Users\\rashid\\Desktop\\customer_churn_prediction\\data\\processed\\x_train.csv', index=False)
x_test_final.to_csv('C:\\Users\\rashid\\Desktop\\customer_churn_prediction\\data\\processed\\x_test.csv', index=False)
y_train.to_csv('C:\\Users\\rashid\\Desktop\\customer_churn_prediction\\data\\processed\\y_train.csv', index=False)
y_test.to_csv('C:\\Users\\rashid\\Desktop\\customer_churn_prediction\\data\\processed\\y_test.csv', index=False)